# Qwen3-8B SFT Fine-Tuning — Berlin TV Tower Persona
**Dataset:** `berlin_tv_tower_extended_training.jsonl` (local)  
**Model:** `unsloth/Qwen3-8B` (BF16, full precision)  
**Framework:** Unsloth + TRL SFTTrainer  
**Goal:** Fine-tune Qwen3-8B to respond as the Berlin TV Tower (Fernsehturm)

## 1. Environment Check

In [ ]:
import torch

print(f"Torch version:      {torch.__version__}")
print(f"CUDA version:       {torch.version.cuda}")
print(f"CUDA available:     {torch.cuda.is_available()}")
print(f"GPU:                {torch.cuda.get_device_name(0)}")
print(f"VRAM:               {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Model + Tokenizer

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME = "unsloth/Qwen3-8B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM after load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 3. Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,} total)")

## 4. Load + Prepare Dataset

Local JSONL file — each line is:  
`{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}`

In [ ]:
from datasets import load_dataset

DATASET_PATH = "berlin_tv_tower_training.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.shuffle(seed=42)

print(f"Total examples: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print("\nExample entry:")
print(dataset[0])

In [ ]:
# 90/10 split — small dataset so training gets the majority
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,  # No reasoning trace — cleaner persona responses
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(format_chat,   remove_columns=val_dataset.column_names)

print("Formatted example:")
print(train_dataset[0]["text"])

## 5. Baseline Inference (Before Fine-Tuning)

The base model will respond generically. After training it should speak in first person as the Fernsehturm.

In [ ]:
from unsloth import FastLanguageModel as FLM

def run_inference(model, tokenizer, prompt, max_new_tokens=256):
    FLM.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

# Prompts designed to test persona fidelity
TEST_PROMPTS = [
    "What is your name?",
    "How tall are you?",
    "What can I see from the top of you?",
    "When were you built?",
    "Do you have a restaurant inside you?",
]

baseline_outputs = {}
for prompt in TEST_PROMPTS:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    response = run_inference(model, tokenizer, prompt)
    baseline_outputs[prompt] = response
    print(response)

## 6. Training

Small dataset → more epochs, lower LR, packing off.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

training_args = SFTConfig(
    output_dir="./qwen3-8b-tv-tower-lora",

    # --- Core hyperparams ---
    num_train_epochs=5,             
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4, 
    learning_rate=1e-4,             
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # --- Sequence ---
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                  

    # --- Logging ---
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",

    # --- Misc ---
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print(f"Train examples:       {len(train_dataset)}")
print(f"Steps per epoch:      {len(trainer.get_train_dataloader())}")
print(f"Total steps:          {len(trainer.get_train_dataloader()) * training_args.num_train_epochs}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
train_result = trainer.train()

print("\n--- Training Summary ---")
print(f"Train loss:      {train_result.training_loss:.4f}")
print(f"Train runtime:   {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:     {train_result.metrics['train_samples_per_second']:.1f}")

## 7. After Fine-Tuning Inference

Same prompts as baseline — the model should now respond in first person as the Fernsehturm.

In [ ]:
finetuned_outputs = {}
for prompt in TEST_PROMPTS:
    response = run_inference(model, tokenizer, prompt)
    finetuned_outputs[prompt] = response

# Side-by-side comparison
for prompt in TEST_PROMPTS:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'-'*60}")
    print(f"BEFORE:\n{baseline_outputs[prompt]}")
    print(f"{'-'*60}")
    print(f"AFTER:\n{finetuned_outputs[prompt]}")

## 8. Free-form Chat with the Tower

Try any prompt to explore the persona interactively.

In [ ]:
custom_prompt = "What do you think about Alexanderplatz?"

print(f"PROMPT: {custom_prompt}")
print("-" * 60)
print(run_inference(model, tokenizer, custom_prompt, max_new_tokens=512))

## 9. Save LoRA Adapter

In [ ]:
SAVE_PATH = "./qwen3-8b-tv-tower-lora-adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to {SAVE_PATH}")

## 10. (Optional) Merge + Export

In [ ]:
# Merge LoRA into base weights — needed for vLLM / Ollama deployment

model.save_pretrained_merged(
    "./qwen3-8b-tv-tower-merged",
    tokenizer,
    save_method="merged_16bit"
)

## 11. VRAM Summary

In [ ]:
allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0) / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"VRAM allocated: {allocated:.2f} GB")
print(f"VRAM reserved:  {reserved:.2f} GB")
print(f"VRAM total:     {total:.2f} GB")
print(f"VRAM free:      {total - reserved:.2f} GB")